# Bits and Bytes in C

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Read, set, clear and flip bits with &, |, ^, ~ and shifts, on unsigned fixed-width types such as uint32_t.
- Pack several small fields into one integer and unpack them again with masks and shifts.
- Look at the bytes and bits of any value, such as a float, by copying it into an integer with memcpy.

**In this notebook**, one idea at a time (run each cell before moving on):
1. Seeing the bits
2. &, |, ^ and ~: bit by bit
3. Shifts: << and >>
4. Set, clear, flip and test one bit
5. Masks: reading a field of bits
6. Fixed-width types
7. The bytes of a value
8. Common mistakes

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check_int and check_str print ✓ or ✗; the exercises below use them
void check_int(const char *what, long actual, long expected) {
    if (actual == expected) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected %ld, got %ld\n", what, expected, actual);
    }
}

void check_str(const char *what, const char *actual, const char *expected) {
    if (actual == NULL) {
        actual = "(NULL)";
    }
    if (strcmp(actual, expected) == 0) {
        printf("✓ %s\n", what);
    } else {
        printf("✗ %s: expected \"%s\", got \"%s\"\n", what, expected, actual);
    }
}

## 1. Seeing the bits

Every integer is stored as bits. C lets you write numbers in hexadecimal (`0x2A`), where each hex digit is exactly 4 bits, and print them with `%X`. `%08X` pads with zeros to 8 digits. There's no `printf` code for binary, so the cell below defines a small `print_bits`.

In [ ]:
#include <stdint.h>
#include <stdio.h>

void print_bits(uint8_t v) {
    for (int i = 7; i >= 0; i--) {
        putchar((v >> i) & 1 ? '1' : '0');
    }
}

int main(void) {
    uint8_t values[] = {5, 0x2A, 255, 128};
    for (int i = 0; i < 4; i++) {
        printf("%3d = 0x%02X = ", values[i], values[i]);
        print_bits(values[i]);
        printf("\n");
    }
    return 0;
}

🧪 **Try it:** add `0x0F` and `0xF0` to the array: each hex digit is one group of 4 bits.

## 2. &, |, ^ and ~: bit by bit

These operators work on each pair of bits separately:

| Operator | Bit is 1 when… | Typical use |
|---|---|---|
| `a & b` (and) | both bits are 1 | keep only some bits (a mask) |
| `a \| b` (or) | at least one is 1 | turn bits on |
| `a ^ b` (xor) | the bits differ | flip bits |
| `~a` (not) | the bit was 0 | invert everything |

Don't confuse them with `&&` and `||`, which work on whole true/false values.

In [ ]:
#include <stdint.h>
#include <stdio.h>

void print_bits(uint8_t v) {
    for (int i = 7; i >= 0; i--) {
        putchar((v >> i) & 1 ? '1' : '0');
    }
}

void show(const char *label, uint8_t v) {
    printf("%-6s ", label);
    print_bits(v);
    printf("\n");
}

int main(void) {
    uint8_t a = 0xCA;   // 11001010
    uint8_t b = 0x0F;   // 00001111
    show("a", a);
    show("b", b);
    show("a & b", a & b);
    show("a | b", a | b);
    show("a ^ b", a ^ b);
    show("~a", (uint8_t) ~a);
    return 0;
}

🧪 **Try it:** predict each line bit by bit before running. Then compute `a ^ b ^ b`: what do you get back?

## 3. Shifts: << and >>

`x << n` moves every bit n places to the left, filling with zeros: it multiplies by 2ⁿ. `x >> n` moves them to the right: for unsigned values it divides by 2ⁿ, dropping the remainder. `1u << k` is the number with only bit k on, the building block of every bit trick.

In [ ]:
unsigned x = 13;
printf("%u << 1 = %u, %u << 3 = %u\n", x, x << 1, x, x << 3);
printf("%u >> 1 = %u, %u >> 2 = %u\n", x, x >> 1, x, x >> 2);
for (int k = 0; k < 5; k++) {
    printf("1u << %d = %u\n", k, 1u << k);
}

🧪 **Try it:** check that `x >> 2` equals `x / 4` for a few values of x.

## 4. Set, clear, flip and test one bit

With a shifted 1 as a mask, four one-liners handle a single bit k:

| Goal | Code |
|---|---|
| turn on | `flags \|= 1u << k;` |
| turn off | `flags &= ~(1u << k);` |
| flip | `flags ^= 1u << k;` |
| test | `(flags >> k) & 1` |

This is how programs pack many yes/no settings (permissions, options) into one integer.

In [ ]:
uint32_t flags = 0;
flags |= 1u << 3;          // on
flags |= 1u << 0;          // on
printf("after setting bits 3 and 0: %u\n", (unsigned) flags);
flags ^= 1u << 0;          // flip
flags &= ~(1u << 5);       // off (it was already off)
printf("now: %u, bit 3 is %u, bit 0 is %u\n", (unsigned) flags, (unsigned) ((flags >> 3) & 1), (unsigned) ((flags >> 0) & 1));

🧪 **Try it:** turn on bits 1 and 2 as well, and predict the number before running (2 + 4 + 8).

### ✏️ Exercise: Turn a bit on

Return x with bit k set to 1, using `|` and a shifted 1. A bit that is already on stays on.

In [ ]:
unsigned set_bit(unsigned x, int k) {
    // TODO: OR x with 1u << k
    return x;
}

int main(void) {
    check_int("bit 3 of 0", set_bit(0, 3), 8);
    check_int("already on", set_bit(8, 3), 8);
    check_int("bit 4 of 1", set_bit(1, 4), 17);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
unsigned set_bit(unsigned x, int k) {
    return x | (1u << k);
}

int main(void) {
    check_int("bit 3 of 0", set_bit(0, 3), 8);
    check_int("already on", set_bit(8, 3), 8);
    check_int("bit 4 of 1", set_bit(1, 4), 17);
    return 0;
}
```

</details>

## 5. Masks: reading a field of bits

Hardware and file formats often pack several small numbers into one integer. To read a field of w bits that starts at bit s: **shift right by s** to bring it down, then **AND with a mask of w ones** (`(1u << w) - 1`, such as `0x3F` for 6 bits). The slides do it for `0xABCD`.

#### 🎞️ Reading a bit field with a shift and a mask

![To read bits 5 to 10 of 0xABCD, shift right by 5 to bring them to the bottom, then AND with 0x3F to keep 6 bits: 011110, or 30](shift-mask-steps.svg)

In [ ]:
uint16_t color = 0xABCD;                 // a 16-bit RGB565 color: 5 bits red, 6 green, 5 blue
unsigned red = (color >> 11) & 0x1F;
unsigned green = (color >> 5) & 0x3F;
unsigned blue = color & 0x1F;
printf("red %u, green %u, blue %u\n", red, green, blue);

🧪 **Try it:** put the three fields back together with `(red << 11) | (green << 5) | blue` and check you get `0xABCD`.

### ✏️ Exercise: The low nibble

Return the lowest 4 bits of x (one hex digit) with a mask.

In [ ]:
unsigned low_nibble(unsigned x) {
    // TODO: AND x with 0xF
    return 0;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}

<details>
<summary>Show an answer</summary>

```c
unsigned low_nibble(unsigned x) {
    return x & 0xFu;
}

int main(void) {
    check_int("0xAB", low_nibble(0xAB), 0xB);
    check_int("0x30", low_nibble(0x30), 0);
    check_int("0xFFFF", low_nibble(0xFFFF), 15);
    return 0;
}
```

</details>

## 6. Fixed-width types

An `int` may be 2, 4 or 8 bytes depending on the machine. For bit work, use the types in `<stdint.h>`, whose size is certain: `uint8_t`, `uint16_t`, `uint32_t`, `uint64_t` (and `int8_t`… for signed). Unsigned types wrap around past their maximum, and their shifts are fully defined.

In [ ]:
printf("uint8_t: %zu, uint16_t: %zu, uint32_t: %zu, uint64_t: %zu bytes\n",
       sizeof(uint8_t), sizeof(uint16_t), sizeof(uint32_t), sizeof(uint64_t));
uint8_t small = 250;
small += 10;
printf("250 + 10 in a uint8_t: %u\n", (unsigned) small);
uint64_t big = 1;
big <<= 40;
printf("2^40 = %llu\n", (unsigned long long) big);

🧪 **Try it:** try `big <<= 70;` instead: shifting by at least the width is undefined behavior, and the memory check reports it.

## 7. The bytes of a value

`memcpy` copies a value's raw bytes into an array of `unsigned char`, so you can look at them one by one. On most computers (x86 and ARM), multi-byte numbers are stored **little-endian**: the least significant byte comes first. The same trick reveals a `float`'s own bits.

In [ ]:
uint32_t value = 0x12345678;
unsigned char bytes[4];
memcpy(bytes, &value, sizeof value);
for (int i = 0; i < 4; i++) {
    printf("byte %d: 0x%02X\n", i, bytes[i]);
}
float f = 1.0f;
uint32_t bits;
memcpy(&bits, &f, sizeof bits);
printf("1.0f is stored as 0x%08X\n", (unsigned) bits);

🧪 **Try it:** look at the bits of `-1.0f` and `2.0f`: which bit is the sign?

## 8. Common mistakes

- **`&` instead of `&&`** (or `|` for `||`): `1 & 2` is 0, while `1 && 2` is true.
- **Precedence:** `x & 1 == 0` means `x & (1 == 0)`. Always use parentheses: `(x & 1) == 0`.
- **Shifting signed values:** `-1 >> 1` or `1 << 31` on an `int` depend on the compiler or are undefined. Use unsigned types.
- **Shifting by the width or more:** `1u << 32` on a 32-bit value is undefined.

In [ ]:
unsigned x = 6;
printf("x & 1 is %u, x && 1 is %d\n", x & 1, x && 1);
if ((x & 1) == 0) {
    printf("%u is even\n", x);
}

🧪 **Try it:** remove the parentheses around `x & 1` and read the warning, then the output.

## In short

C works directly on the bits of integers: `&` (and), `|` (or), `^` (xor), `~` (not), `<<` and `>>` (shifts). Use **unsigned** fixed-width types from `<stdint.h>`, such as `uint32_t`: their size is certain, and their shifts are fully defined.

```c
#include <stdio.h>
#include <stdint.h>
#include <string.h>

int main(void) {
    uint32_t flags = 0;
    flags |= 1u << 3;                     // set bit 3
    flags ^= 1u << 0;                     // flip bit 0
    int bit3 = (flags >> 3) & 1;          // read bit 3
    flags &= ~(1u << 0);                  // clear bit 0
    printf("%u %d\n", (unsigned) flags, bit3);   // 8 1

float f = 1.0f;
    uint32_t bits;
    memcpy(&bits, &f, sizeof bits);       // the float's own bits
    printf("0x%08X\n", (unsigned) bits); // 0x3F800000
    return 0;
}
```

A **mask** selects a field: `(value >> 5) & 0x3F` takes the 6 bits that start at bit 5.

**Watch out:** shift only unsigned values, by less than their width: other shifts are undefined behavior or depend on the compiler.

## Readings

- [Bitwise Operations](https://beej.us/guide/bgc/html/split/bitwise-operations.html) (Beej's Guide to C Programming, chapter 24, 4 min): AND, OR, XOR and NOT, then shifts, and the warning about shifting signed values. Short: the operators are the ones you used in CS Fundamentals, now in C.
- [Fixed Width Integer Types](https://beej.us/guide/bgc/html/split/fixed-width-integer-types.html) (Beej's Guide to C Programming, chapter 37, 5 min): Optional: uint8_t, uint32_t and friends, their limits, and the PRIu32-style format specifiers. Sphinx's compilers also print uint32_t with %u, which the challenges use.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Count the Bits** · Easy
- **RGB565 Colors** · Easy
- **Float Anatomy** · Medium
- **CRC-32** · Hard